### Задание 1
В коде ниже реализован `MultiHeadAttention` с возможностью выбора позиционного кодирования между `RoPE` и `ALiBi`. Допишите реализацию методов `_get_alibi_slopes`, `_apply_alibi` для инициализации и применения `ALiBi`. Ориентируйтесь на уже реализованные методы `get_rope_freqs`, `apply_rope` для `RoPE` позиционной кодировки.

In [ ]:
import torch.nn as nn
import math
import torch.nn.functional as F

class MultiHeadSelfAttention(nn.Module):
    def __init__(self, d_model, num_heads, pos_encoding=None, dropout=0.1):
        """
        Multi-Head Self-Attention с поддержкой позиционных кодировок
        
        Args:
            d_model: размерность модели
            num_heads: количество голов внимания
            pos_encoding: тип позиционного кодирования ('alibi', 'rope', None)
            dropout: вероятность dropout
        """
        super(MultiHeadSelfAttention, self).__init__()
        
        assert d_model % num_heads == 0, "скрытая размерность d_model должна быть кратна количеству голов num_heads"
        
        self.d_model = d_model
        self.num_heads = num_heads
        self.head_dim = d_model // num_heads
        self.scale = math.sqrt(self.head_dim)
        self.pos_encoding = pos_encoding
        
        # Линейные преобразования для Q, K, V
        self.wq = nn.Linear(d_model, d_model)
        self.wk = nn.Linear(d_model, d_model)
        self.wv = nn.Linear(d_model, d_model)
        
        # Финальное линейное преобразование
        self.wo = nn.Linear(d_model, d_model)
        
        # Dropout
        self.dropout = nn.Dropout(dropout)
        
        # Инициализация позиционных кодировок
        if pos_encoding == 'alibi':
            self._init_alibi_biases()
        elif pos_encoding == 'rope':
            self._init_rope_frequencies()
    
    def _init_alibi_biases(self):
        """Инициализация ALiBi смещений"""
        # Создаём slopes для каждой головы
        slopes = torch.tensor(self._get_alibi_slopes(self.num_heads))
        self.register_buffer('alibi_slopes', slopes.view(1, self.num_heads, 1, 1))
    
    def _get_alibi_slopes(self, n_heads):
        """Генерация slopes для ALiBi"""
        slopes = torch.ones(n_heads)
        slopes *= 2**(-8/n_heads)
        slopes = torch.cumprod(slopes, dim=0)
        return slopes
            
    def _init_rope_frequencies(self):
        """Инициализация частот для RoPE"""
        freqs = self._get_rope_frequencies()
        self.register_buffer('rope_freqs', freqs)

    def _get_rope_frequencies(self):
        """Генерация частот для RoPE"""
        # base задана по условию
        base = 10000.0
        dim = self.head_dim
        freqs = 1.0 / (base ** (torch.arange(0, dim, 2).float() / dim))
        return freqs
    
    def _apply_rope(self, x):
        """Применение RoPE к тензору"""
        batch_size, num_heads, seq_len, head_dim = x.shape
        
        # Создаём позиционные индексы
        positions = torch.arange(seq_len).float()
        
        # Вычисляем синусоидальные компоненты
        freqs = self.rope_freqs.view(1, 1, 1, head_dim // 2)
        positions = positions.view(1, 1, seq_len, 1)
        
        # Вычисляем углы
        theta = positions * freqs
        cos_theta = torch.cos(theta)
        sin_theta = torch.sin(theta)
        
        # Разделяем на пары для вращения
        x_reshaped = x.view(batch_size, num_heads, seq_len, head_dim // 2, 2)
        x1 = x_reshaped[..., 0]
        x2 = x_reshaped[..., 1]
        
        # Применяем вращение
        x1_rot = x1 * cos_theta - x2 * sin_theta
        x2_rot = x1 * sin_theta + x2 * cos_theta
        
        # Собираем обратно
        x_rot = torch.stack([x1_rot, x2_rot], dim=-1)
        x_rot = x_rot.view(batch_size, num_heads, seq_len, head_dim)
        
        return x_rot
    
    def _apply_alibi(self, scores):
        """Применение ALiBi смещений к scores"""
        batch_size, head_num, seq_len, seq_len = scores.shape

        # создадим torch.tensor с массивом индексов элементов
        positions = torch.arange(seq_len)

        # Рассчитаем матрицу с попарными расстояниями между парами элементов
        relative_positions = positions.view(1, 1, -1) - positions.view(1, -1, 1)
        relative_positions = relative_positions.abs().neg()

        # вычислим alibi biases
        alibi_bias = self._get_alibi_slopes(head_num).reshape(head_num, 1, 1) * relative_positions

        # Добавляем к scores
        scores = scores + alibi_bias
        return scores
    
    def forward(self, x, mask=None):
        """
        Forward pass с поддержкой позиционных кодировок
        """
        batch_size, seq_len, _ = x.shape
        
        # Линейные преобразования для Q, K, V
        Q = self.wq(x)
        K = self.wk(x)
        V = self.wv(x)
        
        # Reshape для multi-head
        Q = Q.view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        K = K.view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        V = V.view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        
        # Применение RoPE к Q и K (если используется)
        if self.pos_encoding == 'rope':
            Q = self._apply_rope(Q)
            K = self._apply_rope(K)
        
        # Вычисление attention scores
        scores = torch.matmul(Q, K.transpose(-2, -1)) / self.scale
        
        # Применение ALiBi (если используется)
        if self.pos_encoding == 'alibi':
            scores = self._apply_alibi(scores)
        
        # Применение маски
        if mask is not None:
            scores = scores.masked_fill(mask == 0, -1e9)
        
        # Softmax и dropout
        attention_weights = F.softmax(scores, dim=-1)
        attention_weights = self.dropout(attention_weights)
        
        # Умножение на V
        context = torch.matmul(attention_weights, V)
        
        # Конкатенация голов
        context = context.transpose(1, 2).contiguous()
        context = context.view(batch_size, seq_len, self.d_model)
        
        # Финальное линейное преобразование
        output = self.wo(context)
        
        return output, attention_weights


def test_position_invariance():
    """Тест на чувствительность к позиции токенов"""
    
    d_model = 64
    num_heads = 4
    
    # Создаём ОДИН токен в РАЗНЫХ позициях
    token = torch.randn(1, 1, d_model)
    fixed_seq = torch.randn(1, 4, d_model)

    # Первая последовательность: токен в начале
    seq1 = torch.cat([token, fixed_seq], dim=1)
    
    # Вторая последовательность: тот же токен в конце
    seq2 = torch.cat([fixed_seq, token], dim=1)
    
    # Тестируем разные кодировки
    for pos_encoding in [None, 'alibi', 'rope']:
        print(f"\n=== Тест позиционной чувствительности ({pos_encoding}) ===")
        
        # Создаём модель с выключенным dropout для детерминированности
        attention = MultiHeadSelfAttention(
            d_model, 
            num_heads, 
            pos_encoding,
            dropout=0.0
        )
        attention.eval()
        
        with torch.no_grad():
            output1, weights1 = attention(seq1)
            output2, weights2 = attention(seq2)
        
        # Сравниваем выходы для нашего токена (в разных позициях)
        token_output1 = output1[:, 0, :]  # Токен в позиции 0
        token_output2 = output2[:, 4, :]  # Тот же токен в позиции 4
        
        diff = torch.abs(token_output1 - token_output2).mean().item()
        print(f"Разница между выходами одного токена в разных позициях: {diff:.8f}")
        
        if pos_encoding is None:
            # Без позиционной кодировки разница должна быть маленькой
            if diff < 1e-6:
                print("Без позиционной кодировки выходы практически одинаковые")
            else:
                print("Без позиционной кодировки есть заметная разница")
        else:
            # С позиционной кодировкой разница должна быть существенной
            if diff > 1e-3:
                print("С позиционной кодировкой выходы различаются (работает!)")
            else:
                print("С позиционной кодировкой нет заметной разницы")

# блок для запуска тестирования
print("=" * 60)
print("ТЕСТИРОВАНИЕ ПОЗИЦИОННЫХ КОДИРОВОК")
print("=" * 60)
    
# тест
test_position_invariance()
    
print("\nВсе тесты завершены!")

ТЕСТИРОВАНИЕ ПОЗИЦИОННЫХ КОДИРОВОК

=== Тест позиционной чувствительности (None) ===
Разница между выходами одного токена в разных позициях: 0.00000002
Без позиционной кодировки выходы практически одинаковые

=== Тест позиционной чувствительности (alibi) ===
Разница между выходами одного токена в разных позициях: 0.02628084
С позиционной кодировкой выходы различаются (работает!)

=== Тест позиционной чувствительности (rope) ===
Разница между выходами одного токена в разных позициях: 0.01370645
С позиционной кодировкой выходы различаются (работает!)

Все тесты завершены!


/var/folders/_4/p98h_g953379cg5tzm5_l5p40000gn/T/ipykernel_11400/2469986486.py:46: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  slopes = torch.tensor(self._get_alibi_slopes(self.num_heads))
